## 🎯 Learning Objectives
* Successfully set up a finetuning environment for instruction-based LLMs.
* Prepare a custom instruction dataset for finetuning.
* Apply LoRA (Low-Rank Adaptation) to efficiently finetune a small language model.
* Utilize the `trl` library's `SFTTrainer` for supervised finetuning.
* Perform basic inference with a finetuned LoRA adapter.


## FT01-L07: Exercise: Finetune a Small Model on a Custom Instruction Dataset

### Task Description

In this exercise, you will finetune a small, open-source language model on a custom instruction dataset using the LoRA technique. The goal is to adapt the model to follow specific instruction patterns, mimicking how you might prepare a model for a domain-specific task.

This exercise emphasizes practical application, using modern tools and techniques prevalent in 2026 for efficient LLM adaptation.

### Requirements

1.  **Model Selection**: Use a small, publicly available instruction-tuned model (e.g., `TinyLlama-1.1B-Chat-v1.0`, `Phi-3-mini-4k-instruct`).
2.  **Dataset**: Create a mock instruction dataset with at least 5-10 examples. Each example should consist of an instruction, an optional input, and an expected output.
3.  **Finetuning Method**: Employ LoRA (Low-Rank Adaptation) for parameter-efficient finetuning. You should load the base model in 4-bit quantization (QLoRA) to simulate resource-constrained environments.
4.  **Framework**: Utilize the `transformers` library for model loading and tokenization, and the `trl` library's `SFTTrainer` for the finetuning process.
5.  **Training Configuration**: Configure `TrainingArguments` for a short training run (e.g., 1-2 epochs, small batch size) suitable for an exercise.
6.  **Inference**: After finetuning, demonstrate how to load the LoRA adapter and perform inference with the finetuned model on a new instruction.

### Evaluation Criteria

*   **Code Correctness**: The provided code runs without errors and successfully completes the finetuning process.
*   **LoRA Application**: LoRA is correctly configured and applied to the base model.
*   **Dataset Preparation**: The custom dataset is correctly formatted and prepared for `SFTTrainer`.
*   **Training Execution**: The `SFTTrainer` is initialized and executed with appropriate arguments.
*   **Inference Demonstration**: The finetuned model can generate a coherent response to a new instruction, demonstrating successful adaptation.
*   **Clarity and Comments**: Code is well-structured, readable, and includes comments explaining key steps and decisions.


In [ ]:
# Install necessary libraries (run once)
# !pip install -q transformers peft accelerate bitsandbytes datasets trl

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
import os

# --- Configuration --- 

# 1. Model Choice
# Using TinyLlama for quick execution in an exercise. Phi-3-mini is another excellent small model.
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 2. Mock Instruction Dataset
# This dataset simulates a domain-specific instruction-following task.
# Each entry has an 'instruction', an optional 'input', and an 'output'.
mock_data = [
    {
        "instruction": "Summarize the following text in one sentence.",
        "input": "The quick brown fox jumps over the lazy dog. This is a classic pangram used to display all letters of the alphabet.",
        "output": "The quick brown fox, a classic pangram, demonstrates all letters of the alphabet."
    },
    {
        "instruction": "Translate the following English sentence to French.",
        "input": "Hello, how are you?",
        "output": "Bonjour, comment allez-vous ?"
    },
    {
        "instruction": "Explain the concept of 'LoRA' in simple terms.",
        "input": None,
        "output": "LoRA (Low-Rank Adaptation) is a technique to efficiently finetune large language models by only training a small number of new parameters, making it much faster and less memory-intensive."
    },
    {
        "instruction": "What is the capital of France?",
        "input": None,
        "output": "The capital of France is Paris."
    },
    {
        "instruction": "Generate a short, positive affirmation.",
        "input": None,
        "output": "I am capable of achieving great things and I embrace every challenge with confidence."
    },
    {
        "instruction": "Convert 100 Fahrenheit to Celsius.",
        "input": None,
        "output": "100 Fahrenheit is 37.78 Celsius."
    },
    {
        "instruction": "List three benefits of regular exercise.",
        "input": None,
        "output": "Regular exercise improves cardiovascular health, boosts mood, and increases energy levels."
    }
]

# Convert the list of dictionaries to a Hugging Face Dataset
dataset = Dataset.from_list(mock_data)

# 3. LoRA Configuration
# These parameters are crucial for efficient finetuning.
LORA_R = 16          # LoRA attention dimension
LORA_ALPHA = 32      # Alpha parameter for LoRA scaling
LORA_DROPOUT = 0.05  # Dropout probability for LoRA layers

# 4. Training Arguments
# These arguments control the training process.
OUTPUT_DIR = "./finetuned_tinyllama_lora"
BATCH_SIZE = 2       # Small batch size for exercise
GRADIENT_ACCUMULATION_STEPS = 1 # Accumulate gradients over this many steps
LEARNING_RATE = 2e-4 # Standard learning rate for LoRA
NUM_TRAIN_EPOCHS = 2 # A small number of epochs for quick demonstration
MAX_SEQ_LENGTH = 512 # Maximum sequence length for tokenization

# --- Helper Function for Dataset Formatting --- 

# This function formats our raw instruction data into a single string
# that the model will learn to generate.
def format_instruction_dataset(example):
    # For instruction-tuned models, a common format is `[INST] instruction [/INST] output`
    # or `### Instruction: ... ### Input: ... ### Response: ...`
    # We'll use a simple chat-like format for TinyLlama-Chat.
    if example.get("input"):
        return {"text": f"<|im_start|>user\n{example['instruction']}\n{example['input']}<|im_end|>\n<|im_start|>assistant\n{example['output']}<|im_end|>"}
    else:
        return {"text": f"<|im_start|>user\n{example['instruction']}<|im_end|>\n<|im_start|>assistant\n{example['output']}<|im_end|>"}

# Apply the formatting function to the dataset
formatted_dataset = dataset.map(format_instruction_dataset)

print("--- Formatted Dataset Example ---")
print(formatted_dataset[0]["text"])
print("---------------------------------")

# --- Model and Tokenizer Loading (Setup for QLoRA) ---

# Quantization configuration for QLoRA
# This loads the model in 4-bit, significantly reducing memory footprint.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Normalized Float 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16, # Compute in bfloat16 for better precision
    bnb_4bit_use_double_quant=True, # Double quantization for even smaller memory footprint
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
# Set padding token to EOS token if not already set, important for batching
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # For causal LMs, padding on the right is common

# Load the base model with 4-bit quantization
# device_map="auto" distributes the model across available GPUs or CPU if no GPU.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 # Ensure model weights are loaded in bfloat16 for computation
)

# Enable gradient checkpointing to save memory during training
model.gradient_checkpointing_enable()

# Set the model to use the tokenizer's padding token for generation
model.config.use_cache = False # Required for gradient checkpointing
model.config.pretraining_tp = 1 # For TinyLlama, helps with distributed training

print(f"Model loaded: {MODEL_NAME}")
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Number of parameters in base model: {model.num_parameters() / 1e6:.2f}M")


### Your Turn: Implement the Finetuning Logic

Now it's your turn to complete the exercise. Using the setup code provided above, implement the finetuning process. Your implementation should:

1.  Define the `LoraConfig` using the parameters provided (`LORA_R`, `LORA_ALPHA`, `LORA_DROPOUT`).
2.  Initialize the `SFTTrainer` from the `trl` library, passing in the model, tokenizer, formatted dataset, LoRA configuration, and training arguments.
3.  Start the training process.
4.  Save the finetuned LoRA adapter.
5.  Load the base model and the saved LoRA adapter to perform inference on a new, unseen instruction.

Feel free to add print statements or visualizations to monitor the training progress or inspect the model's behavior.


In [ ]:
# --- Solution: Finetuning and Inference --- 

# 1. Define LoRA Configuration
# This configuration tells PEFT how to inject LoRA layers into the model.
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none", # Generally recommended to set bias to 'none' for LoRA
    task_type="CAUSAL_LM", # Specify that this is for a causal language model
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Target all linear layers in attention and MLP blocks
)

# 2. Initialize TrainingArguments
# These arguments define the training schedule and logging behavior.
training_arguments = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    logging_steps=10, # Log training metrics every 10 steps
    save_steps=50, # Save checkpoint every 50 steps
    fp16=False, # Set to True if your GPU supports FP16, bfloat16 is used by default with BitsAndBytesConfig
    bf16=True, # Use bfloat16 for training if supported, matches compute_dtype
    optim="paged_adamw_8bit", # Optimized AdamW for 8-bit quantization
    report_to="none", # Don't report to any external services like Weights & Biases
    remove_unused_columns=False, # Keep all columns in dataset for formatting
    max_steps=-1, # Set to -1 to run for num_train_epochs
)

# 3. Initialize SFTTrainer
# SFTTrainer simplifies the process of supervised finetuning.
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=lora_config,
    dataset_text_field="text", # The column in the dataset containing the formatted text
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False, # Set to True for more efficient training with short sequences, but can be complex
)

# 4. Start Training
print("\n--- Starting Finetuning ---")
trainer.train()
print("--- Finetuning Complete ---")

# 5. Save the Finetuned LoRA Adapter
# Only the small LoRA weights are saved, not the entire base model.
trainer.save_model(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

# --- Inference with the Finetuned Model --- 

# Clear GPU memory if needed (optional, but good practice)
del model
del trainer
torch.cuda.empty_cache()

print("\n--- Performing Inference with Finetuned Model ---")

# Load the base model again (or keep it if not deleted)
# Ensure it's loaded with the same quantization config if you want QLoRA inference
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config, # Use the same quantization config
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Load the LoRA adapter onto the base model
# This merges the LoRA weights with the base model for inference.
finetuned_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

# Merge LoRA weights into the base model for easier deployment/inference
# This creates a new model with the LoRA weights baked in.
# For TinyLlama, this is fine. For larger models, you might just keep them separate.
finetuned_model = finetuned_model.merge_and_unload()

# Example new instruction for inference
new_instruction_1 = "What are the main benefits of using LoRA for finetuning LLMs?"
new_instruction_2 = "Summarize the following text: 'Artificial intelligence (AI) is intelligence demonstrated by machines, unlike the natural intelligence displayed by humans and animals. Leading AI textbooks define the field as the study of 'intelligent agents': any device that perceives its environment and takes actions that maximize its chance of successfully achieving its goals.'"

# Format the instruction for the model
def generate_prompt(instruction, input=None):
    if input:
        return f"<|im_start|>user\n{instruction}\n{input}<|im_end|>\n<|im_start|>assistant\n"
    else:
        return f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n"

# Tokenize the prompt
input_prompt_1 = generate_prompt(new_instruction_1)
input_ids_1 = tokenizer(input_prompt_1, return_tensors="pt").input_ids.to(finetuned_model.device)

input_prompt_2 = generate_prompt(new_instruction_2)
input_ids_2 = tokenizer(input_prompt_2, return_tensors="pt").input_ids.to(finetuned_model.device)

# Generate a response
print(f"\nUser: {new_instruction_1}")
with torch.no_grad():
    outputs_1 = finetuned_model.generate(
        input_ids=input_ids_1,
        max_new_tokens=100, # Limit the length of the generated response
        do_sample=True, # Use sampling for more creative responses
        top_p=0.9, # Nucleus sampling
        temperature=0.7, # Controls randomness
        pad_token_id=tokenizer.eos_token_id # Important for generation
    )
response_1 = tokenizer.decode(outputs_1[0][len(input_ids_1[0]):], skip_special_tokens=True)
print(f"Finetuned Model: {response_1.strip()}")

print(f"\nUser: {new_instruction_2}")
with torch.no_grad():
    outputs_2 = finetuned_model.generate(
        input_ids=input_ids_2,
        max_new_tokens=50, # Limit the length of the generated response
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )
response_2 = tokenizer.decode(outputs_2[0][len(input_ids_2[0]):], skip_special_tokens=True)
print(f"Finetuned Model: {response_2.strip()}")

print("\n--- Inference Complete ---")
